# Uber Pickups — Unsupervised Machine Learning

## objective

### Uber wants to recommend **hot-zones** to drivers: geographical areas where demand is likely to be high at a given time.

## 1. Imports and configuration

#### I will keep the workflow reproducible and reusable rather than writing a separate analysis for every day/hour combination.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from zipfile import ZipFile
from io import BytesIO

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.model_selection import ParameterGrid
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

# The notebook expects the uploaded project ZIP to be in the same folder as the notebook.
DATA_ZIP = Path("C:/Users/mickt/Downloads/uber-trip-data(1).zip")

print("Ready.")
print("Dataset:", DATA_ZIP)


Ready.
Dataset: C:\Users\mickt\Downloads\uber-trip-data(1).zip


## 2. Load the Uber data

#### The goal here is to build one consistent dataset covering the available NYC observations. The raw columns are `Date/Time`, `Lat`, `Lon`, and `Base`.

In [2]:
def load_csv_from_zip(zf, filename):
    """Read one CSV stored inside a ZIP archive."""
    with zf.open(filename) as f:
        return pd.read_csv(f, encoding="latin1")


def load_uber_data(zip_path):
    frames = []

    with ZipFile(zip_path) as outer:
        members = [
            m for m in outer.namelist()
            if not m.endswith("/")
            and "__MACOSX" not in m
        ]

        for member in members:
            lower = member.lower()

            if lower.endswith(".csv"):
                df = load_csv_from_zip(outer, member)
                if {"Date/Time", "Lat", "Lon"}.issubset(df.columns):
                    frames.append(df)

            elif lower.endswith(".csv.zip"):
                # Handle nested monthly archive(s), if present.
                nested_bytes = outer.read(member)
                with ZipFile(BytesIO(nested_bytes)) as nested:
                    for nested_member in nested.namelist():
                        if (
                            not nested_member.endswith("/")
                            and "__MACOSX" not in nested_member
                            and nested_member.lower().endswith(".csv")
                        ):
                            df = pd.read_csv(
                                nested.open(nested_member),
                                encoding="latin1"
                            )
                            if {"Date/Time", "Lat", "Lon"}.issubset(df.columns):
                                frames.append(df)

    if not frames:
        raise ValueError("No Uber pickup CSV files were found.")

    data = pd.concat(frames, ignore_index=True)
    return data


df_raw = load_uber_data(DATA_ZIP)

print(f"Rows loaded: {len(df_raw):,}")
print(f"Columns: {list(df_raw.columns)}")
display(df_raw.head())


Rows loaded: 4,534,327
Columns: ['Date/Time', 'Lat', 'Lon', 'Base']


,Date/Time,Lat,Lon,Base
0,4/1/2014 0:11:00,40.7690,-73.9549,B02512
1,4/1/2014 0:17:00,40.7267,-74.0345,B02512
2,4/1/2014 0:21:00,40.7316,-73.9873,B02512
3,4/1/2014 0:28:00,40.7588,-73.9776,B02512
4,4/1/2014 0:33:00,40.7594,-73.9722,B02512


### Initial inspection

Before cleaning, we inspect dimensions, data types, missing values and duplicate rows.

This is important because clustering is sensitive to bad coordinates: a single invalid latitude/longitude can create a completely artificial geographical cluster.

In [3]:
print("Shape:", df_raw.shape)
print("\nData types:")
display(df_raw.dtypes)

print("\nMissing values:")
display(df_raw.isna().sum().to_frame("missing"))

print("\nDuplicate rows:", df_raw.duplicated().sum())

print("\nDescriptive statistics:")
display(df_raw.describe(include="all").T)


Shape: (4534327, 4)

Data types:


Date/Time     object
Lat          float64
Lon          float64
Base          object
dtype: object


Missing values:


,missing
Date/Time,0
Lat,0
Lon,0
Base,0



Duplicate rows: 82581

Descriptive statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Date/Time,4534327,260093,4/7/2014 20:21:00,97,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Lat,4534327.0,NaN,NaN,NaN,40.739261,0.03995,39.6569,40.7211,40.7422,40.761,42.1166
Lon,4534327.0,NaN,NaN,NaN,-73.973019,0.057267,-74.929,-73.9965,-73.9834,-73.9653,-72.0666
Base,4534327,5,B02617,1458853,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Data cleaning and transformation

We now:
1. Parse the pickup timestamp.
2. Convert coordinates to numeric values.
3. Remove missing values and duplicate observations.
4. Remove impossible geographic coordinates.
5. Restrict the analysis to the New York City area represented by the dataset.
6. Create temporal features for hour and day of week.

For the clustering problem, **latitude and longitude are the core spatial variables**. Time variables are used to slice the data into meaningful demand periods rather than being mixed directly into the spatial distance calculation.

In [4]:
df = df_raw.copy()

# Timestamp and numeric conversion
df["Date/Time"] = pd.to_datetime(df["Date/Time"], errors="coerce")
df["Lat"] = pd.to_numeric(df["Lat"], errors="coerce")
df["Lon"] = pd.to_numeric(df["Lon"], errors="coerce")

# Remove invalid/missing observations
df = df.dropna(subset=["Date/Time", "Lat", "Lon"]).copy()
df = df.drop_duplicates().copy()

# Broad NYC bounds for robust outlier filtering.
# The exact limits are intentionally generous enough to retain NYC observations
# while removing obvious GPS/data-entry errors.
NYC_LAT_MIN, NYC_LAT_MAX = 40.55, 40.95
NYC_LON_MIN, NYC_LON_MAX = -74.30, -73.65

df = df[
    df["Lat"].between(NYC_LAT_MIN, NYC_LAT_MAX)
    & df["Lon"].between(NYC_LON_MIN, NYC_LON_MAX)
].copy()

# Temporal features
df["date"] = df["Date/Time"].dt.date
df["hour"] = df["Date/Time"].dt.hour
df["day_of_week"] = df["Date/Time"].dt.day_name()

day_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]
df["day_of_week"] = pd.Categorical(
    df["day_of_week"],
    categories=day_order,
    ordered=True
)

print(f"Clean rows: {len(df):,}")
print(f"Rows removed: {len(df_raw) - len(df):,}")

display(df.head())


Clean rows: 4,427,847
Rows removed: 106,480


,Date/Time,Lat,Lon,Base,date,hour,day_of_week
0,2014-04-01 00:11:00,40.7690,-73.9549,B02512,2014-04-01,0,Tuesday
1,2014-04-01 00:17:00,40.7267,-74.0345,B02512,2014-04-01,0,Tuesday
2,2014-04-01 00:21:00,40.7316,-73.9873,B02512,2014-04-01,0,Tuesday
3,2014-04-01 00:28:00,40.7588,-73.9776,B02512,2014-04-01,0,Tuesday
4,2014-04-01 00:33:00,40.7594,-73.9722,B02512,2014-04-01,0,Tuesday


### Cleaning result

#### The cleaned dataset is now suitable for geographical clustering.

#### The key principle is that we should not let invalid GPS points determine our hot-zones. We therefore validate the spatial range before fitting any clustering model.

#### Next, we inspect demand over time. This helps us understand why a single static map is not enough for Uber: demand patterns can change substantially throughout the week.

In [5]:
daily_counts = (
    df["day_of_week"]
    .value_counts()
    .reindex(day_order)
    .fillna(0)
    .astype(int)
)

fig = px.bar(
    x=daily_counts.index,
    y=daily_counts.values,
    labels={"x": "Day of week", "y": "Number of pickups"},
    title="Uber pickups by day of week"
)
fig.show()

hourly_counts = df["hour"].value_counts().sort_index()

fig = px.line(
    x=hourly_counts.index,
    y=hourly_counts.values,
    markers=True,
    labels={"x": "Hour of day", "y": "Number of pickups"},
    title="Uber pickups by hour of day"
)
fig.show()


## 4. First geographical view: raw pickups

#### Before applying machine learning, we visualize the pickup coordinates.

#### This is our baseline. The points show where pickups occurred, but the raw map can become difficult to interpret when thousands of observations overlap.

#### Clustering will compress these observations into a smaller number of representative geographical zones.

In [6]:
# Sample the points only for visualization so the interactive map remains responsive.
plot_sample = df.sample(
    min(30000, len(df)),
    random_state=RANDOM_STATE
)

fig = px.scatter_map(
    plot_sample,
    lat="Lat",
    lon="Lon",
    zoom=10,
    center={"lat": 40.75, "lon": -73.98},
    map_style="open-street-map",
    title="Raw Uber pickup locations — NYC",
    hover_data=["Date/Time", "Base"]
)
fig.update_layout(height=700)
fig.show()


## 5. Build a reusable clustering dataset

The project brief recommends starting small.

We will first analyze a **single day and hour**, then generalize the method.

The helper below extracts the latitude/longitude observations for any selected day and hour. This makes it possible to reuse exactly the same methodology for every day of the week and every hour.

In [7]:
def get_period_data(data, day, hour):
    subset = data[
        (data["day_of_week"] == day)
        & (data["hour"] == hour)
    ][["Lat", "Lon"]].dropna().copy()

    return subset


# Example starting point: Monday morning at 08:00
example_day = "Monday"
example_hour = 8

period_df = get_period_data(df, example_day, example_hour)

print(f"{example_day} at {example_hour:02d}:00")
print(f"Pickup observations: {len(period_df):,}")

X = period_df[["Lat", "Lon"]].to_numpy()


Monday at 08:00
Pickup observations: 28,628


## 6. K-Means: choosing the number of clusters

#### K-Means requires us to choose the number of clusters.

#### Rather than selecting an arbitrary value, we test a range of possible values. For each K we calculate:
- #### Inertia: how compact the clusters are.
- #### Silhouette score: how well separated and internally cohesive the clusters are.
- #### Davies–Bouldin score: lower values indicate better separation/compactness.



In [8]:
from sklearn.model_selection import ParameterGrid

def grid_search_kmeans(X, k_values=range(2, 16)):
    results = []

    for params in ParameterGrid({"n_clusters": list(k_values)}):
        model = KMeans(
            n_clusters=params["n_clusters"],
            random_state=RANDOM_STATE,
            n_init=10
        )
        labels = model.fit_predict(X)

        results.append({
            "n_clusters": params["n_clusters"],
            "inertia": model.inertia_,
            "silhouette": silhouette_score(X, labels),
            "davies_bouldin": davies_bouldin_score(X, labels)
        })

    return pd.DataFrame(results)


k_results = grid_search_kmeans(X)

display(k_results)

fig = px.line(
    k_results,
    x="n_clusters",
    y="inertia",
    markers=True,
    title="K-Means elbow analysis",
    labels={
        "n_clusters": "Number of clusters (K)",
        "inertia": "Inertia"
    }
)
fig.show()

fig = px.line(
    k_results,
    x="n_clusters",
    y="silhouette",
    markers=True,
    title="K-Means silhouette score by K",
    labels={
        "n_clusters": "Number of clusters (K)",
        "silhouette": "Silhouette score"
    }
)
fig.show()


,n_clusters,inertia,silhouette,davies_bouldin
0,2,62.534377,0.629897,0.688812
1,3,42.909692,0.411905,0.800017
2,4,31.825670,0.440006,0.640273
3,5,24.216769,0.468209,0.658330
4,6,17.501204,0.483363,0.595622
5,7,15.124283,0.397153,0.665235
6,8,13.021366,0.416621,0.671833
7,9,11.214261,0.423683,0.673076
8,10,10.204804,0.376131,0.736375
9,11,9.367433,0.395501,0.736244


### Selecting K

We use the grid-search results rather than hard-coding a value.

The default selection below maximizes the silhouette score. In a real Uber deployment, the final K could also incorporate operational constraints—for example, how many zones drivers can reasonably be shown at once.

We keep the selected value visible so it can be challenged and discussed during the presentation.

In [9]:
best_k = int(
    k_results.loc[k_results["silhouette"].idxmax(), "n_clusters"]
)

print("Best K according to silhouette score:", best_k)

kmeans = KMeans(
    n_clusters=best_k,
    random_state=RANDOM_STATE,
    n_init=10
)

period_df = period_df.copy()
period_df["cluster"] = kmeans.fit_predict(
    period_df[["Lat", "Lon"]]
)

centers = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=["Lat", "Lon"]
)

cluster_sizes = (
    period_df["cluster"]
    .value_counts()
    .sort_index()
    .rename("pickup_count")
)

centers["pickup_count"] = cluster_sizes.values
centers["cluster"] = centers.index

display(centers.sort_values("pickup_count", ascending=False))


Best K according to silhouette score: 2


,Lat,Lon,pickup_count,cluster
1,40.742413,-73.983094,26192,1
0,40.755266,-73.858867,2436,0


## 7. K-Means hot-zone map

Each K-Means centroid is a candidate **hot-zone**.

The cluster size tells us how many pickups belong to that zone during the selected period. This is useful from a business perspective because Uber could prioritize larger/high-demand zones when recommending positioning to drivers.

In [10]:
fig = px.scatter_map(
    period_df,
    lat="Lat",
    lon="Lon",
    color="cluster",
    zoom=10,
    center={"lat": 40.75, "lon": -73.98},
    map_style="open-street-map",
    title=f"K-Means pickup clusters — {example_day} at {example_hour:02d}:00",
    opacity=0.45
)

fig.add_trace(
    go.Scattermap(
        lat=centers["Lat"],
        lon=centers["Lon"],
        mode="markers+text",
        text=[
            f"Hot-zone {c}<br>{n:,} pickups"
            for c, n in zip(centers["cluster"], centers["pickup_count"])
        ],
        textposition="top center",
        marker=dict(size=18),
        name="Hot-zone centers"
    )
)

fig.update_layout(height=750)
fig.show()


## 8. DBSCAN

#### K-Means assumes that clusters can be represented by centroids and requires the number of clusters in advance.

#### DBSCAN takes a different approach: it identifies **dense regions of points** and can label isolated observations as noise.

In [11]:
def grid_search_dbscan(
    X,
    eps_values=np.arange(0.001, 0.010, 0.001),
    min_samples_values=(20, 40, 60, 80)
):
    results = []

    for params in ParameterGrid({
        "eps": list(eps_values),
        "min_samples": list(min_samples_values)
    }):
        model = DBSCAN(
            eps=params["eps"],
            min_samples=params["min_samples"]
        )

        labels = model.fit_predict(X)

        non_noise = labels != -1
        n_clusters = len(set(labels[non_noise]))

        # Silhouette is only meaningful when at least two non-noise
        # clusters remain after removing DBSCAN noise.
        if n_clusters >= 2 and non_noise.sum() > n_clusters:
            sil = silhouette_score(X[non_noise], labels[non_noise])
            db = davies_bouldin_score(X[non_noise], labels[non_noise])
        else:
            sil = np.nan
            db = np.nan

        results.append({
            "eps": params["eps"],
            "min_samples": params["min_samples"],
            "n_clusters": n_clusters,
            "noise_ratio": 1 - non_noise.mean(),
            "silhouette": sil,
            "davies_bouldin": db
        })

    return pd.DataFrame(results)


dbscan_results = grid_search_dbscan(X)

display(
    dbscan_results
    .sort_values("silhouette", ascending=False)
    .head(15)
)


,eps,min_samples,n_clusters,noise_ratio,silhouette,davies_bouldin
3,0.001,80,14,0.890981,0.838120,0.209027
2,0.001,60,48,0.771238,0.714900,0.370869
31,0.008,80,4,0.067696,0.659484,0.235707
34,0.009,60,4,0.045829,0.638005,0.245490
33,0.009,40,6,0.036712,0.526296,0.270501
35,0.009,80,5,0.051174,0.408564,0.358310
30,0.008,60,5,0.050161,0.405641,0.358356
1,0.001,40,96,0.487180,0.371394,0.727727
29,0.008,40,6,0.041009,0.363731,0.357999
25,0.007,40,6,0.047296,0.351138,0.397603


### Selecting DBSCAN parameters

#### We prioritize configurations that create at least two clusters and provide a strong silhouette score, while avoiding an excessive proportion of observations classified as noise.

In [12]:
valid_db = dbscan_results.dropna(subset=["silhouette"]).copy()

if valid_db.empty:
    raise ValueError(
        "No DBSCAN configuration produced at least two non-noise clusters. "
        "Expand the eps/min_samples grid."
    )

best_dbscan_row = valid_db.sort_values(
    ["silhouette", "noise_ratio"],
    ascending=[False, True]
).iloc[0]

best_eps = float(best_dbscan_row["eps"])
best_min_samples = int(best_dbscan_row["min_samples"])

print("Selected DBSCAN eps:", best_eps)
print("Selected DBSCAN min_samples:", best_min_samples)
print("Expected clusters:", int(best_dbscan_row["n_clusters"]))
print("Noise ratio:", round(float(best_dbscan_row["noise_ratio"]) * 100, 2), "%")

dbscan = DBSCAN(
    eps=best_eps,
    min_samples=best_min_samples
)

period_df["dbscan_cluster"] = dbscan.fit_predict(
    period_df[["Lat", "Lon"]]
)

display(
    period_df["dbscan_cluster"]
    .value_counts()
    .sort_index()
    .rename("pickup_count")
    .to_frame()
)


Selected DBSCAN eps: 0.001
Selected DBSCAN min_samples: 80
Expected clusters: 14
Noise ratio: 89.1 %


,pickup_count
dbscan_cluster,
-1,25507
0,532
1,101
2,627
3,186
4,97
5,175
6,127
7,567


## 9. DBSCAN hot-zone map

Cluster `-1` represents DBSCAN noise.

Unlike K-Means, DBSCAN does not force every point into a cluster. The remaining dense clusters represent areas where pickups are geographically concentrated.

The comparison between the two maps helps us assess which definition of "hot-zone" is more useful for the business.

In [13]:
db_plot = period_df.copy()

fig = px.scatter_map(
    db_plot,
    lat="Lat",
    lon="Lon",
    color="dbscan_cluster",
    zoom=10,
    center={"lat": 40.75, "lon": -73.98},
    map_style="open-street-map",
    title=f"DBSCAN pickup clusters — {example_day} at {example_hour:02d}:00",
    opacity=0.45
)

fig.update_layout(height=750)
fig.show()


## 10. Generalize the K-Means approach across the week

The project requires us to describe hot-zones **per day of the week**.

Instead of manually repeating the analysis, we create a reusable function. For each day/hour combination, the function:
1. extracts pickup coordinates,
2. performs the K-Means K search,
3. selects K using silhouette score,
4. fits the final model,
5. calculates cluster centers,
6. ranks zones by pickup count.

This turns the one-period prototype into a framework that can support an eventual driver-facing recommendation system.

In [14]:
def fit_best_kmeans_for_period(
    data,
    day,
    hour,
    k_min=2,
    k_max=10,
    min_points=100
):
    subset = get_period_data(data, day, hour)

    if len(subset) < min_points:
        return None, None, None

    X_period = subset[["Lat", "Lon"]].to_numpy()

    max_k = min(k_max, len(subset) - 1)
    if max_k < k_min:
        return None, None, None

    results = grid_search_kmeans(
        X_period,
        k_values=range(k_min, max_k + 1)
    )

    best_k = int(
        results.loc[results["silhouette"].idxmax(), "n_clusters"]
    )

    model = KMeans(
        n_clusters=best_k,
        random_state=RANDOM_STATE,
        n_init=10
    )

    subset["cluster"] = model.fit_predict(X_period)

    centers = pd.DataFrame(
        model.cluster_centers_,
        columns=["Lat", "Lon"]
    )
    centers["cluster"] = centers.index
    centers["pickup_count"] = (
        subset["cluster"]
        .value_counts()
        .sort_index()
        .values
    )
    centers["day_of_week"] = day
    centers["hour"] = hour
    centers["k"] = best_k

    return model, subset, centers


# Example: run the complete process for every hour of Monday.
monday_centers = []

for hour in range(24):
    _, _, centers = fit_best_kmeans_for_period(
        df, "Monday", hour
    )
    if centers is not None:
        monday_centers.append(centers)

monday_centers = (
    pd.concat(monday_centers, ignore_index=True)
    if monday_centers
    else pd.DataFrame()
)

display(monday_centers.head(20))


,Lat,Lon,cluster,pickup_count,day_of_week,hour,k
0,40.735290,-73.976302,0,5525,Monday,0,3
1,40.661141,-73.791566,1,591,Monday,0,3
2,40.699104,-74.181605,2,167,Monday,0,3
3,40.735033,-73.981196,0,3458,Monday,1,2
4,40.691512,-73.804659,1,201,Monday,1,2
5,40.740667,-73.994491,0,1739,Monday,2,4
6,40.797611,-73.935576,1,368,Monday,2,4
7,40.700965,-73.808091,2,98,Monday,2,4
8,40.690460,-73.955095,3,657,Monday,2,4
9,40.739722,-73.981614,0,5710,Monday,3,2


## 11. Hot-zones by day of week

The following analysis runs K-Means for a chosen hour on all seven days. This makes the temporal comparison easy to explain: the geographical demand pattern can move depending on the day.

In [15]:
def get_hot_zones_by_day(
    data,
    hour,
    k_min=2,
    k_max=10,
    min_points=100
):
    all_centers = []

    for day in day_order:
        _, _, centers = fit_best_kmeans_for_period(
            data,
            day,
            hour,
            k_min=k_min,
            k_max=k_max,
            min_points=min_points
        )

        if centers is not None:
            all_centers.append(centers)

    if not all_centers:
        return pd.DataFrame()

    return pd.concat(all_centers, ignore_index=True)


analysis_hour = 18

hot_zones_week = get_hot_zones_by_day(
    df,
    hour=analysis_hour
)

display(
    hot_zones_week
    .sort_values(["day_of_week", "pickup_count"], ascending=[True, False])
    .head(30)
)


,Lat,Lon,cluster,pickup_count,day_of_week,hour,k
8,40.741289,-73.985079,0,51064,Friday,18,2
9,40.725196,-73.833930,1,2426,Friday,18,2
0,40.742320,-73.985159,0,33500,Monday,18,2
1,40.709511,-73.826588,1,2729,Monday,18,2
10,40.739319,-73.982763,0,42701,Saturday,18,2
11,40.713164,-73.821720,1,1950,Saturday,18,2
12,40.737337,-73.985043,0,24143,Sunday,18,2
13,40.707008,-73.823309,1,3414,Sunday,18,2
6,40.742196,-73.986128,0,51023,Thursday,18,2
7,40.734514,-73.845494,1,3569,Thursday,18,2


### Interpreting the weekly result

Each point on the following map is a cluster center. Larger pickup counts indicate zones that attracted more historical pickups during the selected hour.

For a real product, Uber could calculate these zones continuously and use them as driver positioning recommendations.

The important business insight is not only *where* the hot-zones are, but also that they can change by time and day.

In [16]:
if not hot_zones_week.empty:
    fig = px.scatter_map(
        hot_zones_week,
        lat="Lat",
        lon="Lon",
        color="day_of_week",
        size="pickup_count",
        hover_data=["day_of_week", "hour", "pickup_count", "k"],
        zoom=10,
        center={"lat": 40.75, "lon": -73.98},
        map_style="open-street-map",
        title=f"K-Means hot-zone centers by day — {analysis_hour:02d}:00"
    )
    fig.update_layout(height=750)
    fig.show()
else:
    print("No periods met the minimum number of observations.")


## 13. Model comparison
### K-Means
#### - Gives a fixed number of representative centers.
#### - Produces easy-to-explain hot-zones.
#### - Works well when the objective is to position drivers around a limited number of demand centers.

### DBSCAN
#### - Finds dense areas automatically.
#### - Can identify noise.
#### - Does not require a predefined number of zones.
#### - Can represent irregularly shaped geographical concentrations.

#### For Uber's specific use case, the best model is not necessarily the one with the highest mathematical score. We should also consider interpretability, stability, computational cost, and operational usefulness.

In [18]:
# Quantitative comparison on the initial example period

comparison_rows = []

# K-Means metrics
km_labels = period_df["cluster"].to_numpy()
comparison_rows.append({
    "algorithm": "K-Means",
    "clusters": len(np.unique(km_labels)),
    "noise_ratio": 0.0,
    "silhouette": silhouette_score(X, km_labels),
    "davies_bouldin": davies_bouldin_score(X, km_labels)
})

# DBSCAN metrics
db_labels = period_df["dbscan_cluster"].to_numpy()
db_non_noise = db_labels != -1
db_clusters = len(np.unique(db_labels[db_non_noise]))

if db_clusters >= 2:
    comparison_rows.append({
        "algorithm": "DBSCAN",
        "clusters": db_clusters,
        "noise_ratio": 1 - db_non_noise.mean(),
        "silhouette": silhouette_score(
            X[db_non_noise],
            db_labels[db_non_noise]
        ),
        "davies_bouldin": davies_bouldin_score(
            X[db_non_noise],
            db_labels[db_non_noise]
        )
    })

comparison = pd.DataFrame(comparison_rows)

display(comparison)


,algorithm,clusters,noise_ratio,silhouette,davies_bouldin
0,K-Means,2,0.000000,0.629897,0.688812
1,DBSCAN,14,0.890981,0.838120,0.209027


## 14. Final business conclusion

### What we have demonstrated

We transformed historical Uber pickup data into a form suitable for unsupervised geographical analysis.

We then:
- cleaned invalid observations,
- extracted temporal features,
- investigated pickup demand by day/hour,
- tested multiple K-Means values systematically,
- tuned DBSCAN parameters,
- identified geographical clusters,
- represented K-Means cluster centers as potential hot-zones,
- and created interactive Plotly maps.

### Business interpretation

The resulting clusters provide Uber with a way to answer:

Instead of relying on a single static map, Uber can build a day × hour demand map and update recommendations according to historical pickup patterns.

### Recommended approach

K-Means is particularly attractive when Uber wants a predictable number of easy-to-understand driver zones. DBSCAN is valuable as a complementary method because it detects dense areas and handles noise naturally.